In [ ]:
from sctoolbox.utils.jupyter import bgcolor, _compare_version

# change the background of input cells
bgcolor("PowderBlue", select=[2, 5, 8, 17, 22, 24, 26, 28])

nb_name = "multiomics.ipynb"

_compare_version(nb_name)

# Multiomics Analysis
<hr style="border:2px solid black"> </hr>

## 1 - Description
**Requires two anndata object with precomputed clusters.**

This notebook performs multiome analysis on two single-cell modalities (e.g. RNA and ATAC) 
using [muon](https://github.com/scverse/muon), a tool providing the interface to handle multiome data, and [MOFA+](https://biofam.github.io/MOFA2/).

Starting from two AnnData objects, one per modality, the notebook guides through the following steps:

**Modality Integration**
   - Combining both modalities into a MuData object
   - Barcode matching and alignment between modalities

**Dimensionality Reduction**
   - Compare individual UMAPs and Clustering between modalities
   - Generate joint dimensionality reduction using MOFA+

**Clustering & Embedding**
   - Clustering of cells based on MOFA+ factors
   - UMAP embedding for visualization

**Saving MuData and AnnData**
   - Saving MuData object
   - Save individual modality adatas with joined clustering

### 1.1 How to run a multiomics analysis

The optimal route for this notebook would be to analize both modalities individually by using notebooks 1-4.  
Then both modalities can be merged with this notebook, which also generates a joined clustering.  
This clustering will then be saved back into the individual adatas. With the individual adatas containing the joined clustering the downstream analyis can be done individually for each modality.  
With the downstream analyis done you can input both adatas into the CELLxGENE preparation notebook (`general_notebooks/prepare_for_cellxgene.ipynb`) to generate one joined adata suited for a CELLxGENE deployment.

---

## 2 - Setup

In [ ]:
import sctoolbox.utils as utils
import sctoolbox.tools as tools
import sctoolbox.plotting as pl
from sctoolbox import settings

import scanpy as sc
import pandas as pd
import muon as mu
from pathlib import Path

mu.set_options(pull_on_update=True)

settings.settings_from_config("config.yaml", key="multiomics")

with pd.option_context("display.max.rows", None, "display.max_colwidth", None):
    display(utils.general.get_version_report(report="versions.yml"))

---

## 3 - General Input

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Modality 1
path_mod1 = "/workspace/single_cell/sav47/ATAC/adatas/anndata_4.h5ad"  # Path to adata of modality 2
modality1_prefix = "ATAC"

# Modality 2
path_mod2 = "/workspace/single_cell/sav47/RNA/Analysis/run_1/adatas/anndata_4.h5ad"  # Path to adata of modality 1
modality2_prefix = "RNA"

# Path to barcode mapping file
# See beelow for an example file
barcode_mapping_file = "/mnt/agnerds/user/stefan.guenther/multiome_bc/multiome-arc-dict.txt"  
sample_delimiter = "-"

The barcode mapping file contains a table with two columns without any header.

- The first column contains the barcodes of modality 1.
- The second column contains the matching barcodes of modality 2.

Example:

|||
|:---|:---|
|TAGCCCTGATATTGGG|AGGTCTTAGTGAACCT|
|GGATCCAGATATTGGG|AGGTCTTAGTGAACGA|
|...|...|

---

## 4 - Load anndatas
<hr style="border:2px solid black"> </hr>

### 4.1 Load modality 1 and set individual parameters

In [ ]:
adata_modality_1 = sc.read_h5ad(path_mod1)


In [ ]:
with pd.option_context("display.max.rows", 5, "display.max.columns", None):
    display(adata_modality_1)
    display(adata_modality_1.obs)
    display(adata_modality_1.var)

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
modality1_clustering_column = "leiden_custom"
sample_mod1 = "Sample"

---

### 4.1 Load modality 2 and set individual parameters

In [ ]:
adata_modality_2 = sc.read_h5ad(path_mod2)

In [ ]:
with pd.option_context("display.max.rows", 5, "display.max.columns", None):
    display(adata_modality_2)
    display(adata_modality_2.obs)
    display(adata_modality_2.var)

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
modality2_clustering_column = "clustering"
sample_mod2 = "Sample"

---

## 5 - Join modalities
<hr style="border:2px solid black"> </hr>

### 5.1 - Match barcodes

Each cell in the dataset is uniquely identified by a barcode sequence. However, the two modalities 
(e.g. RNA and ATAC) may use different barcodes to identify the same cell. In this step, we match 
the barcodes so that each cell is represented by the same barcode across modalities, 
allowing us to confidently link the modality measurements from the same cell together.

In [ ]:
a1_matched, a2_matched = tools.multiomics.match_barcodes(adata_modality_1,
                                                         adata_modality_2,
                                                         barcode_mapping_file,
                                                         inplace=False)

---

### 5.2 - Join

Modality spanning barcodes allow to combine the AnnDatas into a single MuData object 
using the `join_modalities` function. MuData is a container format built on top of AnnData, 
designed to store and manage multimodal single-cell datasets. Each modality retains its own 
feature space and metadata, while shared cell-level information is accessible across both modalities. 
This unified structure is the basis for all downstream multiome analyses.

In [ ]:
mdata = tools.multiomics.join_modalities(a1_matched,
                                         a2_matched,
                                         modality1_prefix,
                                         modality2_prefix)

---

### 5.3 - Cluster comparison

### 5.3.1 - Embedding comparison

To evaluate the correspondence between the two modalities, we generate a series of UMAP visualizations 
that allow us to compare the clustering results across modalities. We start with an overview of each 
modality's own clustering, followed by cross-modality projections where cells are colored by the cluster 
assignment of the opposing modality. This gives a first impression of how well the clusters align between 
modalities.

In [ ]:
_ = pl.multiomics.visualize_cluster_comparison(mdata,
                                               modality1_clustering_column,
                                               modality2_clustering_column,
                                               save="multiomics_cluster_comparison.pdf")

### 5.3.2 Calculate similarty matrices

For a more quantifiable comparison similarity matrices are calculated.

In [ ]:
dfs_heatmap, dfs_mods, dfs_sankey = tools.multiomics.compare_clusters(mdata,
                                                                      modality1_clustering_column,
                                                                      modality2_clustering_column)

### 5.3.3 - Comparison heatmap

To quantify the correspondence between the two modalities, we generate a heatmap that shows the overlap 
between the clusters of both modalities. Each field in the heatmap represents the percentage of barcodes (cells) 
from one modality's cluster that are shared with a cluster of the other modality. This allows us to 
identify which clusters in one modality correspond to which clusters in the other modality, and to what 
degree. A high percentage in a given field indicates a strong correspondence between the two clusters, 
while a low percentage suggests little to no overlap.

In [ ]:
_ = pl.multiomics.compare_cluster_heatmap(mdata,
                                          dfs_heatmap,
                                          save="multiomics_cluster_comparison.pdf")

### 5.3.4 - Sankey plot

To further visualize the correspondence between the two modalities, we generate a Sankey diagram that 
shows the flow of cells between the clusters of both modalities. Each node represents a cluster from 
one of the modalities, and each connection represents the number of cells shared between two clusters. 
The width of the connections is proportional to the number of shared cells, making it easy to identify 
the dominant cluster relationships at a glance. Together with the heatmap, this provides a comprehensive 
overview of the cluster correspondence between the two modalities.

In [ ]:
_ = pl.multiomics.plot_sankey(dfs_sankey[0],
                              [modality1_prefix, modality2_prefix],
                              [modality1_clustering_column, modality2_clustering_column],
                              save="cluster_overlap_sankey.html")

### 5.3.5 - Modality Cluster Grid

Next, we generate a grid of UMAP plots that shows: 
The clusters of one modality, paired with every cluster of the other modality. In each plot, the shared 
cells between the two clusters are highlighted, while all other cells are grayed out. The plot with the 
best matching cluster of the opposing modality is framed to identify shared identities across modalities. This grid visualization provides the most granular view of the cluster relationships and complements the heatmap and Sankey diagram.

In [ ]:
_ = pl.multiomics.visualize_modality_grid(mdata,
                                          [modality1_clustering_column, modality2_clustering_column],
                                          save="Modality_1_cluster_grid.pdf")

In [ ]:
_ = pl.multiomics.visualize_modality_grid(mdata,
                                          [modality1_clustering_column, modality2_clustering_column],
                                          use_mod_1=False,
                                          save="Modality_2_cluster_grid.pdf")

---

## 6 - Multi-omics Factor Analysis
<hr style="border:2px solid black"> </hr>

> "Multi-omic factor analysis (MOFA) is a group factor analysis method that allows to learn an interpretable latent space jointly on multiple modalities. Intuitively, it can be viewed as a generalisation of PCA for multi-omics data. More information about this method can be found on the [MOFA website](https://biofam.github.io/MOFA2/)."  

\- [Muon wiki](https://muon.readthedocs.io/en/latest/omics/multi.html)

<div class="alert alert-block alert-danger">
Note on Runtime: MOFA+ can take a significant amount of time to run depending on the 
size of the dataset. To avoid having to rerun MOFA+ every time the notebook is executed, a 
checkpoint file is used. If a checkpoint file is found and `use_checkpoint` is set to `True`, 
the MOFA+ results are loaded from the checkpoint file instead of rerunning MOFA+. If no 
checkpoint file is found or `use_checkpoint` is set to `False`, MOFA+ is run from scratch 
and a checkpoint file is written upon successful completion for future use.
</div>

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
use_checkpoint = True  # Use checkpoint file if True

---

In [ ]:
# Checkpoint file
checkpoint = "mudata_mofa_checkpoint.h5mu"
# Set checkpoint as Path to enable checks
checkpoint_file = Path(f"{settings.adata_output_dir}/{checkpoint}")
if use_checkpoint and not checkpoint_file.is_file():
    print("No checkpoint found.")
    use_checkpoint = False

### 6.1 - Run MOFA+

In [ ]:
if use_checkpoint:
    mdata = mu.read(checkpoint_file)
else:
    mu.tl.mofa(mdata)

    # Write checkpoint
    mdata.write(checkpoint_file)

---

### 6.2 - Calculate UMAP and find the best setting

Similar to the analysis in notebook 3 and 4 we will calculate the neighbors and try to find the best embedding by iterating over multiple UMAP parameters.

In [ ]:
mu.pl.mofa(mdata, color=[f"{modality1_prefix}:{modality1_clustering_column}", f"{modality2_prefix}:{modality2_clustering_column}"])

In [ ]:
sc.pp.neighbors(mdata, use_rep="X_mofa")

Find best UMAP setting. After visually inspecting the results, adjust the parameters shown below for the best embedding. While it is somewhat subjective what the "best" parameters for an embedding should be, the chosen embedding should display clear structures that are neither spread too thin nor too clumped up.

**Parameter overview**

|Parameter|Description|
|---------|-----------|
|`min_dist`|Distances between points to make the plot look more 'clustered'.|
|`spread`|The effective scale of embedded points. Relative to `min_dist`|

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
dist_range = (0.1, 0.31, 0.1)  # Set min_dist range for umap
spread_range = (1.0, 2.5, 0.5)  # Set spread range for umap

---

In [ ]:
_ = pl.multiomics.umap_parameter_sweep(mdata,
                                       min_dist_range=dist_range,
                                       spread_range=spread_range)

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Final choice of spread / dist for umap
min_dist = 0.4
spread = 2.5

---

In [ ]:
# Calculate final UMAP
sc.tl.umap(mdata, min_dist=min_dist, spread=spread)

# Plot UMAP showing modality clustering
sc.pl.umap(mdata, color=[f"{modality1_prefix}:{modality1_clustering_column}",
                         f"{modality2_prefix}:{modality2_clustering_column}"])

---

### 6.3 - Clustering

This step assigns each cell into a cluster. Cells in the same cluster are assumed to be of the same cell type. Cells are assigned based on their distance within the nearest neighbor graph, which is loosely equivalent to their distance within the embedding. The resolution controls the coarseness of the clustering. A lower resolution results in fewer larger clusters, while a higher resolution results in more smaller clusters.

- `clustering_column`: To choose a resolution, change the number in `leiden_0.5`, for example `leiden_0.1` for a resolution of `0.1`. Higher values lead to more clusters.

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
cluster_res_range = (0.1, 1, 0.1)  # Set the searched resolution range from low to high resolution (less to more clusters).
cluster_ncols = 4  # Number of columns displayed in the plot

---

In [ ]:
_ = pl.clustering.search_clustering_parameters(
    mdata,
    ncols=cluster_ncols,
    resolution_range=cluster_res_range,
    report=True,
    save="clustering_search.png")

<h1><center>⬐ Fill in input data here ⬎</center></h1>

In [ ]:
# Choose final resolution
clustering_column = "leiden_0.5"

---

In [ ]:
# Create final clustering
mdata.obs["clustering"] = utils.tables.rename_categories(mdata.obs[clustering_column])

In [ ]:
# Plot final leiden
sc.pl.umap(mdata, color=[f"{modality1_prefix}:{modality1_clustering_column}",
                         f"{modality2_prefix}:{modality2_clustering_column}",
                         "clustering"])

---

# 7 - Save muData and annDatas

In [ ]:
tools.multiomics.export_modality_adatas(mdata)

In [ ]:
mdata.write(f"{settings.adata_output_dir}/mudata.h5mu")